In [33]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error
import numpy as np
from tqdm import tqdm
import pandas as pd


In [34]:
train_df_all_features = pd.read_csv("data/processed_data/train.csv")
test_df_all_features = pd.read_csv("data/processed_data/test.csv")


In [35]:
unique_test_vessel_ids = test_df_all_features["vesselId"].unique()

models = {}

best_features_for_model = {}

best_model_params = {}

In [36]:
lat_model_features = ['time', 'latitude_1_steps_ago','latitude_2_steps_ago', 'latitude_3_steps_ago','latitude_4_steps_ago', 'latitude_5_steps_ago', 'linreg_predicted_latitude', 'longitude_1_steps_ago','longitude_2_steps_ago', 'longitude_3_steps_ago', 'longitude_4_steps_ago', 'longitude_5_steps_ago','linreg_predicted_longitude', 'is_moving', 'time_diff_gt_1day']
long_model_features = ['time', 'latitude_1_steps_ago','latitude_2_steps_ago', 'latitude_3_steps_ago','latitude_4_steps_ago', 'latitude_5_steps_ago', 'linreg_predicted_latitude', 'longitude_1_steps_ago','longitude_2_steps_ago', 'longitude_3_steps_ago', 'longitude_4_steps_ago', 'longitude_5_steps_ago','linreg_predicted_longitude', 'is_moving', 'time_diff_gt_1day']

feature_search_space = ['time', 'latitude_1_steps_ago',
       'longitude_1_steps_ago', 'time_position_1_steps_ago',
       'latitude_2_steps_ago', 'longitude_2_steps_ago',
       'time_position_2_steps_ago', 'latitude_3_steps_ago',
       'longitude_3_steps_ago', 'time_position_3_steps_ago',
       'latitude_4_steps_ago', 'longitude_4_steps_ago',
       'time_position_4_steps_ago', 'latitude_5_steps_ago',
       'longitude_5_steps_ago', 'time_position_5_steps_ago',
       'max_lat_change_last_5_steps', 'min_lat_change_last_5_steps',
       'avg_lat_change_last_5_steps', 'max_long_change_last_5_steps',
       'min_long_change_last_5_steps', 'avg_long_change_last_5_steps',
       'month_of_the_year', 'week_of_the_year', 'day_of_the_year',
       'day_of_the_month', 'day_of_the_week', 'hour_of_the_day',
       'hours_passed', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
       'time_diff_gt_10min', 'time_diff_gt_20min', 'time_diff_gt_40min',
       'time_diff_gt_1hour', 'time_diff_gt_2hours', 'time_diff_gt_6hours',
       'time_diff_gt_12hours', 'time_diff_gt_1day',
       'linreg_predicted_latitude', 'linreg_predicted_longitude',
       'port_lat', 'port_long', 'port_based_predicted_latitude',
       'port_based_predicted_longitude', 'avg_linreg_port_predicted_latitude',
       'avg_linreg_port_predicted_longitude']

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_squared_error
import json
from tqdm import tqdm
from scipy.stats import uniform, randint

def neg_rmse_scorer(y_true, y_pred):
    """
    Calculate negative RMSE so that maximizing this score 
    minimizes the actual RMSE
    """
    return -np.sqrt(mean_squared_error(y_true, y_pred))

def perform_random_param_search(train_df_all_features, unique_vessel_ids, lat_model_features, long_model_features):
    # Parameter distributions for random search
    param_distributions = {
        'learning_rate': uniform(0.05, 0.15),
        'max_depth': randint(8, 14),
        'n_estimators': randint(400, 600),
        'min_child_weight': randint(1, 4),
        'subsample': uniform(0.8, 0.2),
        'colsample_bytree': uniform(0.8, 0.2)
    }

    best_params_dict = {}
    
    # Create scorer - we use greater_is_better=True since we're maximizing negative RMSE
    rmse_scorer_obj = make_scorer(neg_rmse_scorer, greater_is_better=True)

    # Number of random combinations to try
    n_iter = 40

    # Iterate through each vessel
    for vessel_id in tqdm(unique_vessel_ids, desc="Processing vessels"):
        # Get data for this vessel
        vessel_data = train_df_all_features[train_df_all_features["vesselId"] == vessel_id]
              
            
        # Prepare features and targets
        X_lat = vessel_data[lat_model_features]
        y_lat = vessel_data["latitude"]
        X_long = vessel_data[long_model_features]
        y_long = vessel_data["longitude"]
        
        # Initialize models
        lat_model = XGBRegressor(random_state=42)
        long_model = XGBRegressor(random_state=42)
        
        # Perform RandomizedSearch for latitude model
        lat_random_search = RandomizedSearchCV(
            estimator=lat_model,
            param_distributions=param_distributions,
            n_iter=n_iter,
            scoring=rmse_scorer_obj,
            cv=3,
            random_state=42,
            n_jobs=-1,
            verbose=0
        )
        
        # Perform RandomizedSearch for longitude model
        long_random_search = RandomizedSearchCV(
            estimator=long_model,
            param_distributions=param_distributions,
            n_iter=n_iter,
            scoring=rmse_scorer_obj,
            cv=3,
            random_state=42,
            n_jobs=-1,
            verbose=0
        )
        
        try:
            # Fit both models
            lat_random_search.fit(X_lat, y_lat)
            long_random_search.fit(X_long, y_long)
            
            # Store best parameters - convert negative scores back to positive RMSE
            best_params_dict[vessel_id] = {
                "latitude_model": {
                    "best_params": lat_random_search.best_params_,
                    "best_score": -float(lat_random_search.best_score_)  # Convert back to positive RMSE
                },
                "longitude_model": {
                    "best_params": long_random_search.best_params_,
                    "best_score": -float(long_random_search.best_score_)  # Convert back to positive RMSE
                }
            }
            
        except Exception as e:
            print(f"Error processing vessel {vessel_id}: {str(e)}")
            continue
    
    # Save results to file
    with open('best_xgboost_params_random.json', 'w') as f:
        json.dump(best_params_dict, f, indent=4)
    
    # Calculate and print average best scores
    lat_scores = [v["latitude_model"]["best_score"] for v in best_params_dict.values()]
    long_scores = [v["longitude_model"]["best_score"] for v in best_params_dict.values()]
    
    print("\nResults Summary:")
    print(f"Average best latitude RMSE: {np.mean(lat_scores):.4f}")
    print(f"Average best longitude RMSE: {np.mean(long_scores):.4f}")
    print(f"Median latitude RMSE: {np.median(lat_scores):.4f}")
    print(f"Median longitude RMSE: {np.median(long_scores):.4f}")
    
    return best_params_dict

train_df_all_features = pd.read_csv("data/processed_data/train.csv")
unique_vessel_ids = test_df_all_features["vesselId"].unique()

lat_model_features = ['time', 'latitude_1_steps_ago','latitude_2_steps_ago', 
                        'latitude_3_steps_ago','latitude_4_steps_ago', 'latitude_5_steps_ago', 
                        'linreg_predicted_latitude', 'longitude_1_steps_ago','longitude_2_steps_ago', 
                        'longitude_3_steps_ago', 'longitude_4_steps_ago', 'longitude_5_steps_ago',
                        'linreg_predicted_longitude', 'is_moving', 'time_diff_gt_1day']

long_model_features = ['time', 'latitude_1_steps_ago','latitude_2_steps_ago', 
                        'latitude_3_steps_ago','latitude_4_steps_ago', 'latitude_5_steps_ago', 
                        'linreg_predicted_latitude', 'longitude_1_steps_ago','longitude_2_steps_ago', 
                        'longitude_3_steps_ago', 'longitude_4_steps_ago', 'longitude_5_steps_ago',
                        'linreg_predicted_longitude', 'is_moving', 'time_diff_gt_1day']

best_params = perform_random_param_search(train_df_all_features, unique_vessel_ids, 
                                            lat_model_features, long_model_features)

In [38]:
# TRAIN WITH TEST SPLIT

from sklearn.model_selection import train_test_split
from tqdm import tqdm

rmse_lat_models = []
rmse_long_models = []

with tqdm(total=len(unique_test_vessel_ids), unit="vesselId") as pbar:
    for vesselId in unique_test_vessel_ids:
        train_df_vesselId = train_df_all_features[train_df_all_features["vesselId"] == vesselId]

        train_X_lat = train_df_vesselId[lat_model_features]
        train_X_long = train_df_vesselId[long_model_features]
        train_y_lat = train_df_vesselId["latitude"]
        train_y_long = train_df_vesselId["longitude"]

        X_train_lat, X_test_lat, y_train_lat, y_test_lat = train_test_split(train_X_lat, train_y_lat, test_size=0.2, random_state=42)
        X_train_long, X_test_long, y_train_long, y_test_long = train_test_split(train_X_long, train_y_long, test_size=0.2, random_state=42)

        xgb_latitude_best_params = {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 1000}
        xgb_longitude_best_params = {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 1000}

        xgb_latitude = XGBRegressor(**xgb_latitude_best_params)
        xgb_longitude = XGBRegressor(**xgb_longitude_best_params)

        xgb_latitude.fit(X_train_lat, y_train_lat)
        xgb_longitude.fit(X_train_long, y_train_long)
        pred_lat = xgb_latitude.predict(X_test_lat)
        pred_long = xgb_longitude.predict(X_test_long)

        rmse_latitude = root_mean_squared_error(y_test_lat, pred_lat)
        rmse_longitude = root_mean_squared_error(y_test_long, pred_long)

        rmse_lat_models.append(rmse_latitude)
        rmse_long_models.append(rmse_longitude)


        models[vesselId] = {
            "lat_model": xgb_latitude,
            "long_model": xgb_longitude
        }

        pbar.update(1)

print("FINAL AVERAGE RESULTS:")
print(f"Average lat model rmse: {np.mean(rmse_lat_models)}")
print(f"Average long model rmse: {np.mean(rmse_long_models)}")
print(f"----------------")
print(f"Median lat model rmse: {np.median(rmse_lat_models)}")
print(f"Median long model rmse: {np.median(rmse_long_models)}")





100%|██████████| 215/215 [03:39<00:00,  1.02s/vesselId]

FINAL AVERAGE RESULTS:
Average lat model rmse: 0.9617365335328095
Average long model rmse: 2.743916821930673
----------------
Median lat model rmse: 0.24551984757154693
Median long model rmse: 0.437541983572797


#### RESULTS:

lat_model_features = ['time','latitude_1_steps_ago', 'time_position_1_steps_ago', 'latitude_2_steps_ago', 'time_position_2_steps_ago', 'latitude_3_steps_ago', 'time_position_3_steps_ago', 'latitude_4_steps_ago', 'time_position_4_steps_ago', 'latitude_5_steps_ago', 'time_position_5_steps_ago', 'linreg_predicted_latitude', 'port_based_predicted_latitude']

long_model_features = ['time','longitude_1_steps_ago', 'time_position_1_steps_ago', 'longitude_2_steps_ago', 'time_position_2_steps_ago', 'longitude_3_steps_ago', 'time_position_3_steps_ago', 'longitude_4_steps_ago', 'time_position_4_steps_ago', 'longitude_5_steps_ago', 'time_position_5_steps_ago', 'linreg_predicted_longitude','port_based_predicted_longitude']

FINAL AVERAGE RESULTS:
Average lat model rmse: 0.9070610023076858
Average long model rmse: 2.534056204063656
----------------
Median lat model rmse: 0.32603192839661677
Median long model rmse: 0.49273711732072834



lat_model_features = ['time','latitude_1_steps_ago','longitude_1_steps_ago', 'time_position_1_steps_ago','week_of_the_year','day_of_the_year',]
long_model_features = ['time','latitude_1_steps_ago','longitude_1_steps_ago', 'time_position_1_steps_ago','week_of_the_year','day_of_the_year',]

FINAL AVERAGE RESULTS:
Average lat model rmse: 1.2628290309328538
Average long model rmse: 2.9501791949770046
----------------
Median lat model rmse: 0.543673942427989
Median long model rmse: 0.8934150113466613


lat_model_features = ['latitude_1_steps_ago','latitude_2_steps_ago','linreg_predicted_latitude', 'is_moving']
long_model_features = ['longitude_1_steps_ago','longitude_2_steps_ago','linreg_predicted_longitude', 'is_moving']

Average lat model rmse: 1.01003076947678
Average long model rmse: 3.1324865598450744
----------------
Median lat model rmse: 0.4039919367817661
Median long model rmse: 0.5696829982797559

lat_model_features = ['time', 'latitude_1_steps_ago','latitude_2_steps_ago','linreg_predicted_latitude', 'port_based_predicted_latitude', 'is_moving', 'time_diff_gt_1day']
long_model_features = ['time', 'longitude_1_steps_ago','longitude_2_steps_ago','linreg_predicted_longitude', 'port_based_predicted_longitude', 'is_moving', 'time_diff_gt_1day']

Average lat model rmse: 0.9432591171281952
Average long model rmse: 2.5613681591507085
----------------
Median lat model rmse: 0.32390816864412664
Median long model rmse: 0.5525888430829814






In [139]:
# TRAIN ON ALL DATA

models = {}


from tqdm import tqdm

rmse_lat_models = []
rmse_long_models = []

with tqdm(total=len(unique_test_vessel_ids), unit="vesselId") as pbar:
    for vesselId in unique_test_vessel_ids:
        train_df_vesselId = train_df_all_features[train_df_all_features["vesselId"] == vesselId]

        train_X_lat = train_df_vesselId[lat_model_features]
        train_X_long = train_df_vesselId[long_model_features]
        train_y_lat = train_df_vesselId["latitude"]
        train_y_long = train_df_vesselId["longitude"]

        xgb_latitude_best_params = {'learning_rate': 0.1, 'max_depth': 12, 'n_estimators': 500}
        xgb_longitude_best_params = {'learning_rate': 0.1, 'max_depth': 12, 'n_estimators': 500}

        xgb_latitude = XGBRegressor(**xgb_latitude_best_params)
        xgb_longitude = XGBRegressor(**xgb_longitude_best_params)

        xgb_latitude.fit(train_X_lat, train_y_lat)
        pred_lat = xgb_latitude.predict(train_X_lat)
        xgb_longitude.fit(train_X_long, train_y_long)
        pred_long = xgb_longitude.predict(train_X_long)

        rmse_latitude = root_mean_squared_error(train_y_lat, pred_lat)
        rmse_longitude = root_mean_squared_error(train_y_long, pred_long)

        rmse_lat_models.append(rmse_latitude)
        rmse_long_models.append(rmse_longitude)


        models[vesselId] = {
            "lat_model": xgb_latitude,
            "long_model": xgb_longitude
        }

        pbar.update(1)

print("FINAL AVERAGE RESULTS:")
print(f"Average lat model rmse: {np.mean(rmse_lat_models)}")
print(f"Average long model rmse: {np.mean(rmse_long_models)}")
print(f"----------------")
print(f"Median lat model rmse: {np.median(rmse_lat_models)}")
print(f"Median long model rmse: {np.median(rmse_long_models)}")





100%|██████████| 215/215 [04:21<00:00,  1.22s/vesselId]

FINAL AVERAGE RESULTS:
Average lat model rmse: 0.00517973215608151
Average long model rmse: 0.0063091188662768435
----------------
Median lat model rmse: 0.00243325997243382
Median long model rmse: 0.003125793145674626


In [142]:
import numpy as np
from sklearn.linear_model import LinearRegression
from tqdm.auto import tqdm

def linreg_predict_position(row):
    """
    Fits lines to historical latitude and longitude data and predicts next positions.
    Works with both pandas Series and named tuples from itertuples().
    
    Args:
        row: A pandas Series or named tuple containing historical position data
        
    Returns:
        tuple: (predicted_latitude, predicted_longitude)
    """
    # Helper function to safely get attribute from either Series or named tuple
    def get_value(obj, key):
        if hasattr(obj, '_fields'):  # Named tuple
            return getattr(obj, key)
        else:  # pandas Series
            return obj[key]
    
    if get_value(row, 'is_moving') == 0:
        return get_value(row, 'latitude_1_steps_ago'), get_value(row, 'longitude_1_steps_ago')
    
    # Extract historical latitudes and longitudes
    lats = np.array([
        get_value(row, 'latitude_5_steps_ago'),
        get_value(row, 'latitude_4_steps_ago'),
        get_value(row, 'latitude_3_steps_ago'),
        get_value(row, 'latitude_2_steps_ago'),
        get_value(row, 'latitude_1_steps_ago')
    ])
    
    longs = np.array([
        get_value(row, 'longitude_5_steps_ago'),
        get_value(row, 'longitude_4_steps_ago'),
        get_value(row, 'longitude_3_steps_ago'),
        get_value(row, 'longitude_2_steps_ago'),
        get_value(row, 'longitude_1_steps_ago')
    ])
    
    # Extract corresponding times
    times = np.array([
        get_value(row, 'time_position_5_steps_ago'),
        get_value(row, 'time_position_4_steps_ago'),
        get_value(row, 'time_position_3_steps_ago'),
        get_value(row, 'time_position_2_steps_ago'),
        get_value(row, 'time_position_1_steps_ago')
    ])
    
    # Reshape for sklearn
    times = times.reshape(-1, 1)
    
    # Fit latitude line
    lat_model = LinearRegression()
    lat_model.fit(times, lats)
    
    # Fit longitude line
    long_model = LinearRegression()
    long_model.fit(times, longs)
    
    # Predict for current time
    current_time = np.array([[get_value(row, 'time')]])
    predicted_lat = lat_model.predict(current_time)[0]
    predicted_long = long_model.predict(current_time)[0]
    
    return predicted_lat, predicted_long

import numpy as np
import pandas as pd

def calculate_max_min_changes(row):
    """
    Calculate maximum, minimum, and average changes in latitude and longitude
    Works with both pandas Series and named tuples from itertuples()
    
    Args:
        row: A pandas Series or named tuple containing position data
        
    Returns:
        pd.Series with calculated changes
    """
    # Helper function to safely get attribute from either Series or named tuple
    def get_value(obj, key):
        if hasattr(obj, '_fields'):  # Named tuple
            return getattr(obj, key)
        else:  # pandas Series
            return obj[key]
    
    # Create arrays of previous values including current value
    lat_values = np.array([
        get_value(row, 'latitude_1_steps_ago'),
        get_value(row, 'latitude_2_steps_ago'),
        get_value(row, 'latitude_3_steps_ago'),
        get_value(row, 'latitude_4_steps_ago'),
        get_value(row, 'latitude_5_steps_ago')
    ])
    
    long_values = np.array([
        get_value(row, 'longitude_1_steps_ago'),
        get_value(row, 'longitude_2_steps_ago'),
        get_value(row, 'longitude_3_steps_ago'),
        get_value(row, 'longitude_4_steps_ago'),
        get_value(row, 'longitude_5_steps_ago')
    ])
    
    # Calculate changes between consecutive values
    lat_changes = np.diff(lat_values)
    long_changes = np.diff(long_values)
    
    # Handle cases with NaN values
    lat_changes = lat_changes[~np.isnan(lat_changes)]
    long_changes = long_changes[~np.isnan(long_changes)]
    
    # Return max and min changes (or NaN if no valid changes)
    return pd.Series({
        'max_lat_change_last_5_steps': np.max(lat_changes) if len(lat_changes) > 0 else np.nan,
        'min_lat_change_last_5_steps': np.min(lat_changes) if len(lat_changes) > 0 else np.nan,
        'avg_lat_change_last_5_steps': np.mean(lat_changes) if len(lat_changes) > 0 else np.nan,
        'max_long_change_last_5_steps': np.max(long_changes) if len(long_changes) > 0 else np.nan,
        'min_long_change_last_5_steps': np.min(long_changes) if len(long_changes) > 0 else np.nan,
        'avg_long_change_last_5_steps': np.mean(long_changes) if len(long_changes) > 0 else np.nan,
    })

def predict_position_with_port(feature_vector):
    """
    Predicts next position considering both recent movement and destination port.
    Takes in a feature vector instead of raw row data.
    
    Args:
        feature_vector: pandas DataFrame with a single row containing all needed features
        
    Returns:
        tuple: (predicted_latitude, predicted_longitude)
    """
    if feature_vector['is_moving'].iloc[0] == 0:
        return feature_vector['latitude_1_steps_ago'].iloc[0], feature_vector['longitude_1_steps_ago'].iloc[0]

    # Current position (actually last known position)
    current_lat = feature_vector['latitude_1_steps_ago'].iloc[0]
    current_long = feature_vector['longitude_1_steps_ago'].iloc[0]
    
    # Calculate direction vector to port
    to_port_lat = feature_vector['port_lat'].iloc[0] - current_lat
    to_port_long = feature_vector['port_long'].iloc[0] - current_long
    
    # Normalize the port direction vector
    port_distance = np.sqrt(to_port_lat**2 + to_port_long**2)
    if port_distance > 0:
        to_port_lat = to_port_lat / port_distance
        to_port_long = to_port_long / port_distance
    
    # Get recent movement vector from averages
    recent_lat_change = feature_vector['avg_lat_change_last_5_steps'].iloc[0]
    recent_long_change = feature_vector['avg_long_change_last_5_steps'].iloc[0]
    
    # Normalize the recent movement vector
    recent_magnitude = np.sqrt(recent_lat_change**2 + recent_long_change**2)
    if recent_magnitude > 0:
        recent_lat_change = recent_lat_change / recent_magnitude
        recent_long_change = recent_long_change / recent_magnitude
    
    # Calculate weight based on distance to port
    max_weight_distance = 1.0
    port_weight = max(0, min(1, port_distance / max_weight_distance))
    recent_weight = 1 - port_weight
    
    # Combine the two vectors with weights
    predicted_direction_lat = (recent_weight * recent_lat_change + 
                             port_weight * to_port_lat)
    predicted_direction_long = (recent_weight * recent_long_change + 
                              port_weight * to_port_long)
    
    # Normalize the final direction
    final_magnitude = np.sqrt(predicted_direction_lat**2 + predicted_direction_long**2)
    if final_magnitude > 0:
        predicted_direction_lat = predicted_direction_lat / final_magnitude
        predicted_direction_long = predicted_direction_long / final_magnitude
    
    # Scale by the recent average movement magnitude to get actual distance
    step_magnitude = np.sqrt(recent_lat_change**2 + recent_long_change**2)
    
    # Calculate final predictions
    predicted_lat = current_lat + predicted_direction_lat * step_magnitude
    predicted_long = current_long + predicted_direction_long * step_magnitude
    
    return predicted_lat, predicted_long

In [143]:
test_X = test_df_all_features[['vesselId', 'ID', 'time', 'port_lat', 'port_long', 'is_moving', 'time_diff_gt_1day']]

for col in lat_model_features:
    if col not in test_X.columns:
        test_X[col] = np.nan

train_df = train_df_all_features.sort_values(by=['vesselId', 'time'])

last_known_positions = train_df.groupby('vesselId').agg({
    "latitude": 'last',
    "longitude": 'last',
    "time": 'last',
    "latitude_1_steps_ago": 'last',
    "latitude_2_steps_ago": 'last',
    "latitude_3_steps_ago": 'last',
    "latitude_4_steps_ago": 'last',
    "latitude_5_steps_ago": 'last',
    "longitude_1_steps_ago": 'last',
    "longitude_2_steps_ago": 'last',
    "longitude_3_steps_ago": 'last',
    "longitude_4_steps_ago": 'last',
    "longitude_5_steps_ago": 'last',
    "time_position_1_steps_ago": 'last',
    "time_position_2_steps_ago": 'last',
    "time_position_3_steps_ago": 'last',
    "time_position_4_steps_ago": 'last',
    "time_position_5_steps_ago": 'last',


}).reset_index()

last_known_dict = last_known_positions.set_index('vesselId').to_dict('index')
results_list = []
unique_vessel_ids = test_df_all_features["vesselId"].unique()

last_known_dict.get("clh6aqawa0007gh0z9h6zi9bo")


/var/folders/9w/2pytp4_s6bj5_ff4zz3vl8ph0000gn/T/ipykernel_10334/620021739.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_X[col] = np.nan
/var/folders/9w/2pytp4_s6bj5_ff4zz3vl8ph0000gn/T/ipykernel_10334/620021739.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_X[col] = np.nan
/var/folders/9w/2pytp4_s6bj5_ff4zz3vl8ph0000gn/T/ipykernel_10334/620021739.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = v

{'latitude': 59.89167,
 'longitude': 21.54685,
 'time': 0.3497249101902449,
 'latitude_1_steps_ago': 59.83316,
 'latitude_2_steps_ago': 59.76388,
 'latitude_3_steps_ago': 59.69588,
 'latitude_4_steps_ago': 59.63337,
 'latitude_5_steps_ago': 59.57721,
 'longitude_1_steps_ago': 21.38489,
 'longitude_2_steps_ago': 21.35317,
 'longitude_3_steps_ago': 21.34225,
 'longitude_4_steps_ago': 21.43237,
 'longitude_5_steps_ago': 21.5409,
 'time_position_1_steps_ago': 0.3496854444950414,
 'time_position_2_steps_ago': 0.3496468958712811,
 'time_position_3_steps_ago': 0.3496073985529245,
 'time_position_4_steps_ago': 0.3495679012345679,
 'time_position_5_steps_ago': 0.3495280876846792}

In [144]:
with tqdm(total=len(test_X), unit="row", desc="Making Predictions") as pbar:

    for vessel_id in unique_vessel_ids:
        df = test_X[test_X["vesselId"] == vessel_id].reset_index(drop=True)

        df = df.sort_values(by=['time'])

        lat_model = models[vessel_id]["lat_model"]
        long_model = models[vessel_id]["long_model"]


        for row in df.itertuples(index=True):
            vessel_ID = row.vesselId
            row_ID = row.ID
            current_time = row.time

            last_entry = last_known_dict.get(vessel_ID)

            if last_entry == None:
                raise ValueError("Last entry not found")
            
            #print(f"Dict entry of vessels prev row: {last_known_dict.get(vessel_ID)}")
            # Data for CURRENT STEP
            latitude_1_steps_ago = last_entry['latitude']
            longitude_1_steps_ago = last_entry['longitude']
            time_position_1_steps_ago = last_entry['time']
            latitude_2_steps_ago = last_entry['latitude_1_steps_ago']
            longitude_2_steps_ago = last_entry['longitude_1_steps_ago']
            time_position_2_steps_ago = last_entry['time_position_1_steps_ago']
            latitude_3_steps_ago = last_entry['latitude_2_steps_ago']
            longitude_3_steps_ago = last_entry['longitude_2_steps_ago']
            time_position_3_steps_ago = last_entry['time_position_2_steps_ago']
            latitude_4_steps_ago = last_entry['latitude_3_steps_ago']
            longitude_4_steps_ago = last_entry['longitude_3_steps_ago']
            time_position_4_steps_ago = last_entry['time_position_3_steps_ago']
            latitude_5_steps_ago = last_entry['latitude_4_steps_ago']
            longitude_5_steps_ago = last_entry['longitude_4_steps_ago']
            time_position_5_steps_ago = last_entry['time_position_4_steps_ago']

            #lat_change_2_to_1_steps = latitude_1_steps_ago - latitude_2_steps_ago
            #lon_change_2_to_1_steps = longitude_1_steps_ago - longitude_2_steps_ago
            #lat_change_3_to_2_steps = latitude_2_steps_ago - latitude_3_steps_ago
            #lon_change_3_to_2_steps = longitude_2_steps_ago - longitude_3_steps_ago
            #lat_change_4_to_3_steps = latitude_3_steps_ago - latitude_4_steps_ago
            #lon_change_4_to_3_steps = longitude_3_steps_ago - longitude_4_steps_ago
            #lat_change_5_to_4_steps = latitude_4_steps_ago - latitude_5_steps_ago
            #lon_change_5_to_4_steps = longitude_4_steps_ago - longitude_5_steps_ago
            #cog_1_step_ago = last_entry["cog"]
            #time_cog_1_step_ago = last_entry["time"]
            #cog_2_steps_ago = last_entry["cog_1_step_ago"]
            #time_cog_2_steps_ago = last_entry["time_cog_1_step_ago"]



            # Assign last known positions from training data
            test_X.at[row.Index, "latitude_1_steps_ago"] = latitude_1_steps_ago
            test_X.at[row.Index, "longitude_1_steps_ago"] = longitude_1_steps_ago
            test_X.at[row.Index, "time_position_1_steps_ago"] = time_position_1_steps_ago
            test_X.at[row.Index, "latitude_2_steps_ago"] = latitude_2_steps_ago
            test_X.at[row.Index, "longitude_2_steps_ago"] = longitude_2_steps_ago
            test_X.at[row.Index, "time_position_2_steps_ago"] = time_position_2_steps_ago
            test_X.at[row.Index, "latitude_3_steps_ago"] = latitude_3_steps_ago
            test_X.at[row.Index, "longitude_3_steps_ago"] = longitude_3_steps_ago
            test_X.at[row.Index, "time_position_3_steps_ago"] = time_position_3_steps_ago
            test_X.at[row.Index, "latitude_4_steps_ago"] = latitude_4_steps_ago
            test_X.at[row.Index, "longitude_4_steps_ago"] = longitude_4_steps_ago
            test_X.at[row.Index, "time_position_4_steps_ago"] = time_position_4_steps_ago
            test_X.at[row.Index, "latitude_5_steps_ago"] = latitude_5_steps_ago
            test_X.at[row.Index, "longitude_5_steps_ago"] = longitude_5_steps_ago
            test_X.at[row.Index, "time_position_5_steps_ago"] = time_position_5_steps_ago
            #test_X.at[row.Index, "lat_change_2_to_1_steps"] = lat_change_2_to_1_steps
            #test_X.at[row.Index, "lon_change_2_to_1_steps"] = lon_change_2_to_1_steps
            #test_X.at[row.Index, "lat_change_3_to_2_steps"] = lat_change_3_to_2_steps
            #test_X.at[row.Index, "lon_change_3_to_2_steps"] = lon_change_3_to_2_steps
            #test_X.at[row.Index, "lat_change_4_to_3_steps"] = lat_change_4_to_3_steps
            #test_X.at[row.Index, "lon_change_4_to_3_steps"] = lon_change_4_to_3_steps
            #test_X.at[row.Index, "lat_change_5_to_4_steps"] = lat_change_5_to_4_steps
            #test_X.at[row.Index, "lon_change_5_to_4_steps"] = lon_change_5_to_4_steps
            #test_X.at[row.Index, "cog_1_step_ago"] = cog_1_step_ago
            #test_X.at[row.Index, "time_cog_1_step_ago"] = time_cog_1_step_ago
            #test_X.at[row.Index, "cog_2_steps_ago"] = cog_2_steps_ago
            #test_X.at[row.Index, "time_cog_2_steps_ago"] = time_cog_2_steps_ago

            #test_X.at[row.Index, 'lat_diff_to_port_1step'] = last_known_latitude - row.port_lat
            #test_X.at[row.Index, 'long_diff_to_port_1step'] = last_known_longitude - row.port_long

            # Get linreg features
            feature_series = pd.Series({
                'time': row.time,
                'latitude_5_steps_ago': latitude_5_steps_ago,
                'latitude_4_steps_ago': latitude_4_steps_ago,
                'latitude_3_steps_ago': latitude_3_steps_ago,
                'latitude_2_steps_ago': latitude_2_steps_ago,
                'latitude_1_steps_ago': latitude_1_steps_ago,
                'longitude_5_steps_ago': longitude_5_steps_ago,
                'longitude_4_steps_ago': longitude_4_steps_ago,
                'longitude_3_steps_ago': longitude_3_steps_ago,
                'longitude_2_steps_ago': longitude_2_steps_ago,
                'longitude_1_steps_ago': longitude_1_steps_ago,
                'time_position_5_steps_ago': time_position_5_steps_ago,
                'time_position_4_steps_ago': time_position_4_steps_ago,
                'time_position_3_steps_ago': time_position_3_steps_ago,
                'time_position_2_steps_ago': time_position_2_steps_ago,
                'time_position_1_steps_ago': time_position_1_steps_ago,
                'is_moving': test_X.at[row.Index, "is_moving"]
            })
            linreg_predicted_latitude, linreg_predicted_longitude = linreg_predict_position(feature_series)
            test_X.at[row.Index, "linreg_predicted_latitude"] = linreg_predicted_latitude
            test_X.at[row.Index, "linreg_predicted_longitude"] = linreg_predicted_longitude
            
            # Calculate changes
            feature_series = pd.Series({
                'latitude_5_steps_ago': latitude_5_steps_ago,
                'latitude_4_steps_ago': latitude_4_steps_ago,
                'latitude_3_steps_ago': latitude_3_steps_ago,
                'latitude_2_steps_ago': latitude_2_steps_ago,
                'latitude_1_steps_ago': latitude_1_steps_ago,
                'longitude_5_steps_ago': longitude_5_steps_ago,
                'longitude_4_steps_ago': longitude_4_steps_ago,
                'longitude_3_steps_ago': longitude_3_steps_ago,
                'longitude_2_steps_ago': longitude_2_steps_ago,
                'longitude_1_steps_ago': longitude_1_steps_ago,
            })
            changes = calculate_max_min_changes(feature_series)
            for col in changes.index:
                test_X.at[row.Index, col] = changes[col]
            
            # Create feature vector with all needed columns
            needed_columns = [
                'latitude_1_steps_ago', 'longitude_1_steps_ago',
                'port_lat', 'port_long',
                'avg_lat_change_last_5_steps', 'avg_long_change_last_5_steps',
                'is_moving'
            ]
            feature_vector = test_X.loc[row.Index, needed_columns].to_frame().T
            feature_vector = feature_vector.apply(pd.to_numeric, errors='coerce')
            
            # Get port-based predictions
            port_based_predicted_latitude, port_based_predicted_longitude = predict_position_with_port(feature_vector)
            
            test_X.at[row.Index, "port_based_predicted_latitude"] = port_based_predicted_latitude
            test_X.at[row.Index, "port_based_predicted_longitude"] = port_based_predicted_longitude


            # Make predictions
            #feature_vector = test_X.loc[row.Index, cog_model_expected_features].to_frame().T
            #feature_vector = feature_vector.apply(pd.to_numeric, errors='coerce')

            #red_cog = cog_model.predict(feature_vector)[0]

            feature_vector_lat = test_X.loc[row.Index, lat_model_features].to_frame().T
            feature_vector_lat = feature_vector_lat.apply(pd.to_numeric, errors='coerce')

            feature_vector_long = test_X.loc[row.Index, long_model_features].to_frame().T
            feature_vector_long = feature_vector_long.apply(pd.to_numeric, errors='coerce')

            #feature_vector['cog'] = pred_cog

            is_moving = test_X.at[row.Index, "is_moving"]

            if is_moving:
                pred_lat = lat_model.predict(feature_vector_lat)[0]
                pred_lon = long_model.predict(feature_vector_long)[0]
            else: 
                pred_lat = latitude_1_steps_ago
                pred_lon = longitude_1_steps_ago


            results_list.append({
                "ID": row_ID,
                "latitude_predicted": pred_lat,
                "longitude_predicted": pred_lon,
            })

            last_known_dict[vessel_ID] = {
                'latitude_1_steps_ago': latitude_1_steps_ago,
                'longitude_1_steps_ago': longitude_1_steps_ago,
                'time_position_1_steps_ago': time_position_1_steps_ago,
                'latitude_2_steps_ago': latitude_2_steps_ago,
                'longitude_2_steps_ago': longitude_2_steps_ago,
                'time_position_2_steps_ago': time_position_2_steps_ago,
                'latitude_3_steps_ago': latitude_3_steps_ago,
                'longitude_3_steps_ago': longitude_3_steps_ago,
                'time_position_3_steps_ago': time_position_3_steps_ago,
                'latitude_4_steps_ago': latitude_4_steps_ago,
                'longitude_4_steps_ago': longitude_4_steps_ago,
                'time_position_4_steps_ago': time_position_4_steps_ago,
                'latitude_5_steps_ago': latitude_5_steps_ago,
                'longitude_5_steps_ago': longitude_5_steps_ago,
                'time_position_5_steps_ago': time_position_5_steps_ago,
                #"lat_change_2_to_1_steps": lat_change_2_to_1_steps,
                #"lon_change_2_to_1_steps": lon_change_2_to_1_steps,
                #"lat_change_3_to_2_steps": lat_change_3_to_2_steps,
                #"lon_change_3_to_2_steps": lon_change_3_to_2_steps,
                #"lat_change_4_to_3_steps": lat_change_4_to_3_steps,
                #"lon_change_4_to_3_steps": lon_change_4_to_3_steps,
                #"lat_change_5_to_4_steps": lat_change_5_to_4_steps,
                #"lon_change_5_to_4_steps": lon_change_5_to_4_steps,
                "latitude": pred_lat,
                "longitude": pred_lon,
                "time": current_time,
                #"cog_1_step_ago": cog_1_step_ago,
                #"time_cog_1_step_ago": time_cog_1_step_ago,
                #"cog_2_steps_ago": cog_2_steps_ago,
                #"time_cog_2_steps_ago": cog_2_steps_ago,
                #"cog": pred_cog,
            }
                
            


            pbar.update(1)
            #print(f"Row: {test_X.iloc[row.Index]}")
            #print(f"Pred lat: {pred_lat}, Pred long: {pred_lon}")
            #print(f"Dict entry of vessel: {last_known_dict.get(vessel_ID)}")

# Convert the results list to a DataFrame and sort by ID
results = pd.DataFrame(results_list).sort_values(by=['ID'])

# Save predictions to CSV
results.to_csv("data/v6.csv", index=False)
                
             


            


Making Predictions:   0%|          | 0/51739 [00:00<?, ?row/s]

/var/folders/9w/2pytp4_s6bj5_ff4zz3vl8ph0000gn/T/ipykernel_10334/2244131648.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_X.at[row.Index, "time_position_1_steps_ago"] = time_position_1_steps_ago
/var/folders/9w/2pytp4_s6bj5_ff4zz3vl8ph0000gn/T/ipykernel_10334/2244131648.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_X.at[row.Index, "time_position_2_steps_ago"] = time_position_2_steps_ago
/var/folders/9w/2pytp4_s6bj5_ff4zz3vl8ph0000gn/T/ipykernel_10334/2244131648.py:64: SettingWithCopy